In [1]:
import cv2
import os
import numpy as np
from matplotlib import pyplot as plt

!pip install split-folders ultralytics mlflow dagshub
import splitfolders
import dagshub
import tensorflow as tf
from tensorflow.keras.applications import (
    VGG19, 
    ResNet50, 
    DenseNet121, 
    MobileNetV2, 
    EfficientNetB0, 
    InceptionV3
)
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Rescaling
from tensorflow.keras.models import Model

dagshub.init("tomato-grading", "afkirmohamad4", mlflow=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.1 MB/s eta 0:00:00ta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 77.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 59.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

E0000 00:00:1779533969.269415      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779533969.386291      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779533970.367360      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779533970.367404      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779533970.367407      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779533970.367409      57 computation_placer.cc:177] computation placer already registered. Please check linka

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=8db7fd1d-148a-40a9-9cf3-21cdf305cdd3&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=074a493f5338a111498ef70d0349280f43d972196efefb476a26e94c6a4cff2b




Output()

Accessing as afkirmohamad4

Initialized MLflow to track repo "afkirmohamad4/tomato-grading"

Repository afkirmohamad4/tomato-grading initialized!

In [2]:
input_folder = '/kaggle/input/datasets/mohamedafkir90/datasettomato/Clean_tomato_dataset'
output_folder = '/kaggle/working/tomato_split' 
models_dir = '/kaggle/working/models'

splitfolders.ratio(input_folder, 
                   output=output_folder, 
                   seed=42, 
                   ratio=(.8, .1, .1), 
                   group_prefix=None, 
                   move=False) 

Copying files: 15408 files [03:02, 84.35 files/s] 


In [7]:
from tensorflow.keras.callbacks import Callback

class MLflowEpochLogger(Callback):
    def on_epoch_end(self, epoch, logs=None):
        if logs is None:
            logs = {}
        mlflow.log_metric("train_loss", logs.get("loss"), step=epoch)
        mlflow.log_metric("train_accuracy", logs.get("accuracy"), step=epoch)
        mlflow.log_metric("val_loss", logs.get("val_loss"), step=epoch)
        mlflow.log_metric("val_accuracy", logs.get("val_accuracy"), step=epoch)

In [9]:
import mlflow


mlflow.set_experiment("Tomato_Quality")


models = {
    'DenseNet121': DenseNet121,
    'MobileNetV2': MobileNetV2,
    'EfficientNetB0': EfficientNetB0,
    'InceptionV3': InceptionV3,
    'ResNet50': ResNet50,
    'VGG19': VGG19
}

train_ds = tf.keras.utils.image_dataset_from_directory(f"{output_folder}/train", image_size=(224, 224))
val_ds = tf.keras.utils.image_dataset_from_directory(f"{output_folder}/val", image_size=(224, 224))

test_ds = tf.keras.utils.image_dataset_from_directory(f"{output_folder}/test", image_size=(224, 224))

for name_model, ModelFunction in models.items():
    with mlflow.start_run(run_name=f"{name_model}_Run"):
        print(f"\n--- Training {name_model} ---\n")
        
        inputs = tf.keras.Input(shape=(224, 224, 3))
        x = Rescaling(1./255)(inputs)
        
        base_model = ModelFunction(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
        base_model.trainable = False
        x = GlobalAveragePooling2D()(base_model.output)
        output = Dense(3, activation='softmax')(x) 
        
        model = Model(inputs=base_model.input, outputs=output)
        model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        
        
        mlflow.log_param("model_name", name_model)
        mlflow.log_param("epochs", 20)
        
        
        model.fit(train_ds, 
                  validation_data=val_ds, 
                  epochs=20, 
                  callbacks=[MLflowEpochLogger()])
        
        os.makedirs(models_dir, exist_ok=True)

        save_path = os.path.join(models_dir, f"{name_model}_tomato.keras")
        model.save(save_path)
        print(f"\n--- Saved {name_model} to {save_path}---\n")

        print(f"\n--- Evaluating {name_model} on Test Data --- \n")
        test_loss, test_accuracy = model.evaluate(test_ds)
        
       
        mlflow.log_metric("test_loss", test_loss)
        mlflow.log_metric("test_accuracy", test_accuracy)
        
        print(f"\n--- {name_model} Final Test Accuracy: {test_accuracy:.4f} --- \n")

Found 12324 files belonging to 3 classes.
Found 1539 files belonging to 3 classes.
Found 1545 files belonging to 3 classes.

--- Training DenseNet121 ---

Epoch 1/20
386/386 ━━━━━━━━━━━━━━━━━━━━ 86s 167ms/step - accuracy: 0.7238 - loss: 0.6844 - val_accuracy: 0.8811 - val_loss: 0.2994
Epoch 2/20
386/386 ━━━━━━━━━━━━━━━━━━━━ 34s 88ms/step - accuracy: 0.8806 - loss: 0.2918 - val_accuracy: 0.9019 - val_loss: 0.2432
Epoch 3/20
386/386 ━━━━━━━━━━━━━━━━━━━━ 33s 87ms/step - accuracy: 0.8994 - loss: 0.2494 - val_accuracy: 0.8863 - val_loss: 0.2599
Epoch 4/20
386/386 ━━━━━━━━━━━━━━━━━━━━ 33s 86ms/step - accuracy: 0.9138 - loss: 0.2151 - val_accuracy: 0.8928 - val_loss: 0.2404
Epoch 5/20
386/386 ━━━━━━━━━━━━━━━━━━━━ 34s 88ms/step - accuracy: 0.9145 - loss: 0.2148 - val_accuracy: 0.9311 - val_loss: 0.1924
Epoch 6/20
386/386 ━━━━━━━━━━━━━━━━━━━━ 34s 87ms/step - accuracy: 0.9247 - loss: 0.1947 - val_accuracy: 0.9194 - val_loss: 0.2153
Epoch 7/20
386/386 ━━━━━━━━━━━━━━━━━━━━ 34s 87ms/step - accuracy